# Observability with OpenTelemetry in Python

This notebook demonstrates how to instrument a Python application using OpenTelemetry to generate traces. We will use the `ConsoleSpanExporter` to view the traces directly in the output.

In [ ]:
# Install necessary packages
%pip install opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp

## 1. Setup OpenTelemetry

First, we need to configure the `TracerProvider` and the `ConsoleSpanExporter`. This sets up the infrastructure to collect and export traces.

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor, ConsoleSpanExporter
from opentelemetry.sdk.resources import Resource

def setup_observability(service_name="my-colab-service"):
    """Sets up OpenTelemetry tracing with a Console Exporter."""
    # Define the resource (service name, etc.)
    resource = Resource(attributes={
        "service.name": service_name
    })

    # Create a TracerProvider
    provider = TracerProvider(resource=resource)
    
    # Create a ConsoleSpanExporter to print traces to stdout
    console_exporter = ConsoleSpanExporter()
    
    # Create a BatchSpanProcessor to batch spans before exporting
    processor = BatchSpanProcessor(console_exporter)
    
    # Add the processor to the provider
    provider.add_span_processor(processor)

    # Set the global TracerProvider
    trace.set_tracer_provider(provider)
    
    # Return a tracer
    return trace.get_tracer(__name__)

# Initialize the tracer
tracer = setup_observability()

## 2. Basic Tracing

Now let's create a simple span. A span represents a unit of work.

In [ ]:
with tracer.start_as_current_span("basic-operation") as span:
    print("Doing some work...")
    # Simulate work
    import time
    time.sleep(0.1)
    print("Work done.")

## 3. Attributes and Events

You can add metadata to spans using attributes and log specific moments using events.

In [ ]:
with tracer.start_as_current_span("operation-with-attributes") as span:
    # Set an attribute
    span.set_attribute("user.id", "12345")
    span.set_attribute("operation.type", "calculation")
    
    print("Calculating...")
    
    # Add an event
    span.add_event("calculation_started")
    
    result = 5 * 5
    
    span.add_event("calculation_finished", attributes={"result": result})
    print(f"Result: {result}")

## 4. Nested Spans

Spans can be nested to show the relationship between different parts of your code (parent-child relationship).

In [ ]:
def child_operation():
    with tracer.start_as_current_span("child-task") as child_span:
        child_span.set_attribute("child.attr", "value")
        print("Inside child operation")
        import time
        time.sleep(0.05)

with tracer.start_as_current_span("parent-task") as parent_span:
    print("Inside parent operation")
    child_operation()
    print("Back in parent operation")

## 5. Error Handling

OpenTelemetry can automatically record exceptions.

In [ ]:
try:
    with tracer.start_as_current_span("risky-operation") as span:
        print("About to do something risky...")
        raise ValueError("Something went wrong!")
except ValueError as e:
    print(f"Caught exception: {e}")
    # The span status is automatically set to Error when an exception escapes the context manager,
    # but since we caught it, we might want to record it manually if we want the span to reflect the error.
    # However, standard practice with `start_as_current_span` is that if you catch it, the span is successful unless you explicitly set status.
    # Let's see how to record it manually:
    with tracer.start_as_current_span("handling-error") as error_span:
        error_span.record_exception(e)
        error_span.set_status(trace.Status(trace.StatusCode.ERROR, str(e)))

## 6. Instrumenting AI Agents

In this section, we will simulate an AI Agent workflow and instrument it with OpenTelemetry. We'll define a simple `Agent` class and run a multi-step workflow (Research -> Remediation).

In [ ]:
# Mock tools for CVE data
def search_nist_cve(query: str):
    """Searches the NIST database for CVEs related to the query."""
    return f"Found NIST CVE-2024-1234 for {query}. Severity: High."

def get_gcp_remediation(cve_id: str):
    """Provides Google Cloud remediation steps for a given CVE."""
    return f"Remediation for {cve_id} on GCP: Update to the latest patch level and enable Cloud Armor."

# Define a simple Agent class to mimic the ADK Agent
class Agent:
    def __init__(self, name, description, tools=None):
        self.name = name
        self.description = description
        self.tools = tools or []

# Instantiate Agents
nist_agent = Agent(
    name="nist_agent",
    description="Agent specialized in finding CVEs from NIST.",
    tools=[search_nist_cve]
)

remediation_agent = Agent(
    name="remediation_agent",
    description="Agent specialized in providing GCP remediation instructions.",
    tools=[get_gcp_remediation]
)

def run_agent(agent, query):
    """Runs a single agent with a query, wrapped in a trace."""
    # Start a span for the agent execution
    with tracer.start_as_current_span(f"run_{agent.name}") as span:
        span.set_attribute("agent.name", agent.name)
        span.set_attribute("agent.query", query)
        
        print(f"[{agent.name}] Starting processing for: {query}")
        
        result = ""
        try:
            # Simulate some processing time
            import time
            time.sleep(0.5)
            
            if agent.tools:
                tool = agent.tools[0]
                # For demonstration, we just call the tool with the query
                tool_result = tool(query)
                result = f"Agent {agent.name} result: {tool_result}"
            else:
                result = f"Agent {agent.name} processed {query}"
                
            span.set_attribute("agent.result", result)
            print(f"[{agent.name}] Finished: {result}")
            return result
        except Exception as e:
            span.record_exception(e)
            print(f"[{agent.name}] Error: {e}")
            return str(e)

# Run the workflow
print("--- Starting Agent Workflow ---")
with tracer.start_as_current_span("agent_workflow") as workflow_span:
    user_query = "CVE-2024-1234"
    
    # Step 1: Research
    research_result = run_agent(nist_agent, user_query)
    
    # Step 2: Remediation
    # Pass the research result as context to the remediation agent
    remediation_result = run_agent(remediation_agent, research_result)
    
    print("\n--- Final Output ---")
    print(remediation_result)